In [1]:
%%time
! pip install --upgrade gradio
!pip install -q langchain langchain-openai faiss-cpu pypdf
!pip install langchain langchain-community
!pip install langchain langchain-huggingface
! pip install bs4
! pip install langchain
! pip install langchain-core
! pip install openai
! pip install -q google-genai openai
! pip install -q langgraph google-genai
! pip install sentence-transformers
! pip install edge-tts

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.3/32.3 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.3/73.3 kB 8.2 MB/s eta 0:00:00
  Attempting uninstall: starlette
    Found existing installation: starlette 0.52.1
    Uninstalling starlette-0.52.1:
      Successfully uninstalled starlette-0.52.1
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 1.14.0
    Uninstalling gradio_client-1.14.0:
      Successfully uninstalled gradio_client-1.14.0
  Attempting uninstall: gradio
    Found existing installation: gradio 5.50.0
    Uninstalling gradio-5.50.0:
      Successfully uninstalled gradio-5.50.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires starlette<1.0.0,>=0.49.1, but you have starlette 1.2.1 whi

In [2]:
%%time
import gradio as gr
import getpass
import os
import bs4
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from openai import OpenAI
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing_extensions import List, TypedDict
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate
from google import genai
from google.genai import types
from google.colab import drive

<timed exec>:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.


CPU times: user 27.3 s, sys: 4.69 s, total: 32 s
Wall time: 40.3 s


In [3]:
drive.mount('/content/drive')

dataset = '/content/drive/MyDrive/corpus_first_100.csv'

Mounted at /content/drive


In [4]:
import pandas as pd
from langchain_core.documents import Document # Import Document class

loader_df = pd.read_csv(dataset)

# Convert DataFrame rows to Document objects
docs = []
for index, row in loader_df.iterrows():
    docs.append(Document(page_content=row['text']))

# Join the page content from the loaded documents into a single string for test_chatbot
financial_chatbot = "\n\n".join([doc.page_content for doc in docs])

In [5]:
financial_chatbot

'I\'m not saying I don\'t like the idea of on-the-job training too, but you can\'t expect the company to do that. Training workers is not their job - they\'re building software. Perhaps educational systems in the U.S. (or their students) should worry a little about getting marketable skills in exchange for their massive investment in education, rather than getting out with thousands in student debt and then complaining that they aren\'t qualified to do anything.\n\nSo nothing preventing false ratings besides additional scrutiny from the market/investors, but there are some newer controls in place to prevent institutions from using them. Under the DFA banks can no longer solely rely on credit ratings as due diligence to buy a financial instrument, so that\'s a plus. The intent being that if financial institutions do their own leg work then *maybe* they\'ll figure out that a certain CDO is garbage or not. Edit: lead in\n\nYou can never use a health FSA for individual health insurance pre

In [6]:
# 2. Setting up Embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print("Setup complete. Your chatbot is ready for testing, Let's Go Isaac!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Setup complete. Your chatbot is ready for testing, Let's Go Isaac!


In [7]:
from google.colab import userdata

In [8]:
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

In [9]:
%%time
llm = ChatOpenAI(model="gpt-4-turbo", temperature=0.8, max_tokens=250)

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

CPU times: user 1.22 s, sys: 81.2 ms, total: 1.3 s
Wall time: 1.38 s


In [10]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

In [11]:
%%time
text_splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=10)
all_splits = text_splitter.split_documents(docs)

CPU times: user 24.9 ms, sys: 957 µs, total: 25.8 ms
Wall time: 26.3 ms


In [12]:
%%time
# Create FAISS index from chunks
db = FAISS.from_documents(all_splits, embeddings)

CPU times: user 2.03 s, sys: 199 ms, total: 2.23 s
Wall time: 10.3 s


In [13]:
import openai
import os # Ensure os is imported for environment variable access
import asyncio # Added for async TTS
import edge_tts # Added for TTS
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

chatbot_prompt = ChatPromptTemplate.from_messages([
     ("system", """You are a financial analyst at a stock firm. You need to answer questions related to some financial review in a  polite manner to a prospective client.\n
      Context:\n# {context}"""),
     ("human", "Question: {question}"),
     # Removed: ("ai", "Answer: "), to prevent incomplete AI responses
])

# New prompt for rephrasing the question using chat history
condense_question_prompt = ChatPromptTemplate.from_messages([
     ("system", "Given a chat history and a follow-up question, rephrase the follow-up question to be a standalone question."),
     MessagesPlaceholder(variable_name="chat_history"),
     ("human", "{question}"),
])

def ask_question(question: str, history=None) -> str:
    chat_history_for_llm = []

    if history:
        for turn in history:
            # Safely handle dictionary format, Gradio message objects, or legacy tuples
            if isinstance(turn, dict):
                role = turn.get("role")
                content = turn.get("content", "")
                if role in ["user", "human"]:
                    chat_history_for_llm.append(HumanMessage(content=content))
                elif role in ["assistant", "ai", "model"]:
                    chat_history_for_llm.append(AIMessage(content=content))
            elif hasattr(turn, "role"):  # If it arrives as a Gradio ChatMessage object
                if turn.role in ["user", "human"]:
                    chat_history_for_llm.append(HumanMessage(content=turn.content))
                elif turn.role in ["assistant", "ai", "model"]:
                    chat_history_for_llm.append(AIMessage(content=turn.content))
            elif isinstance(turn, (list, tuple)) and len(turn) == 2:
                if turn[0]: chat_history_for_llm.append(HumanMessage(content=turn[0]))
                if turn[1]: chat_history_for_llm.append(AIMessage(content=turn[1]))

    standalone_question = question
    if chat_history_for_llm:
        try:
            # Condense follow-up question using chat history
            condense_chain = condense_question_prompt | llm
            standalone_question = condense_chain.invoke({
                "question": question,
                "chat_history": chat_history_for_llm
            }).content
        except Exception as e:
            print(f"Error condensing question: {e}")
            standalone_question = question

    # Retrieve matching vector documents
    try:
        retrieved_docs = db.similarity_search(standalone_question, k=4)
    except NameError:
        retrieved_docs = vector_store.similarity_search(standalone_question, k=4)

    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs if hasattr(doc, 'page_content'))

    # Format the prompt messages
    messages = chatbot_prompt.format_messages(question=standalone_question, context=docs_content)

    # Convert LangChain message formats to raw dictionary maps to prevent SDK validation rejections
    formatted_payload = []
    for msg in messages:
        # Standardize roles into API compatible tags
        role_type = "user" if msg.type == "human" else "system" if msg.type == "system" else "assistant"
        formatted_payload.append({"role": role_type, "content": msg.content})

    # Invoke your model using the safe dictionary array payload
    return llm.invoke(formatted_payload).content


def predict(input, history):
     # This 'predict' function is not used by the multimodal_app or agent_predict, but leaving it as is.
     output = ask_question(input, history)

     history.append((input, output))

     return history, history


# This converts voice to text using OpenAI Whisper
def transcribe_audio(audio_path):
    # Ensure OPENAI_API_KEY is set in environment or Colab secrets
    if not os.environ.get("OPENAI_API_KEY"):
        return "Error: OPENAI_API_KEY environment variable not set."

    client = openai.OpenAI()
    with open(audio_path, "rb") as audio_file:
        transcript = client.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file
        )
    return transcript.text

# Added text-to-speech function
async def _tts(text):
    communicate = edge_tts.Communicate(text, "en-US-AndrewNeural")
    path = "response.mp3"
    await communicate.save(path)
    return path


# This function will handle both text and transcribed voice input
# and manage the chat history for gr.Chatbot. It uses the globally defined ask_question.
# It now expects and returns history in the Gradio Chatbot format (list of lists/tuples).
def predict_for_multimodal(message, history):
    if history is None:
        history = []

    # 1. Compute the text generation answer
    rag_answer = ask_question(message, history)

    # 2. Append history using explicit dictionary format definitions to guarantee Gradio matching
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": rag_answer})

    return history, ""

# This function wraps transcribe_audio and then calls the main predict_for_multimodal and includes TTS
def voice_chat_handler(audio_path, history):
    status_msg = "" # For the status_box
    audio_output_path = None
    user_text = "" # Initialize user_text
    bot_text = "" # Initialize bot_text

    if history is None:
        history = []

    if audio_path is None:
        status_msg = "No audio input received. Please record your question."
        return history, audio_output_path, status_msg

    try:
        # 1. Transcribe audio
        status_msg = "Converting audio to text..."
        user_text = transcribe_audio(audio_path)
        if user_text.startswith("Error:"):
            status_msg = user_text
            history.append({"role": "assistant", "content": status_msg})
            return history, audio_output_path, status_msg

        # 2. Get RAG answer
        status_msg = f"User: {user_text}\nGenerating response..."
        bot_text = ask_question(user_text, history) # Use ask_question here

        # 3. Convert bot's response to audio
        status_msg = "Converting response to speech..."
        audio_output_path = asyncio.run(_tts(bot_text))

        status_msg = "Response generated successfully!"

    except Exception as e:
        status_msg = f"Error in voice processing: {e}"
        bot_text = f"I encountered an error while processing your request: {e}"
        audio_output_path = None

    # 4. Update history
    history.append({"role": "user", "content": user_text})
    history.append({"role": "assistant", "content": bot_text})

    return history, audio_output_path, status_msg


with gr.Blocks() as multimodal_app:
    gr.Markdown("## Multimodal RAG Assistant")

    chatbot = gr.Chatbot(label="Chat History")
    # Added status_box and audio_out components
    status_box = gr.Textbox(label="Voice Status", interactive=False, visible=True)
    audio_out = gr.Audio(label="🔊 Voice Response", autoplay=True, streaming=True) # Added for TTS output

    with gr.Row():
        msg = gr.Textbox(placeholder="Type a question...", scale=4)
        audio_in = gr.Audio(sources="microphone", type="filepath", scale=1)

    msg.submit(
        predict_for_multimodal,
        [msg, chatbot],
        [chatbot, msg] # Clear the text input after submission
    )

    audio_in.stop_recording(
        voice_chat_handler,
        [audio_in, chatbot],
        [chatbot, audio_out, status_box] # Updated outputs for voice_chat_handler
    )

multimodal_app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9b0ce5169529ac5858.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Asynccio is a library used to write conccurreent code using the async/await syntax which allow program to peerform multiple operations at the same time without using multiple threads or processes.

Asyncio event loop watcches and eevent and dispatch them to the approrpaiate asyncio task. If a task need data from aa network it awaits that operation and the event loop swtich to another task

#Multimodal with image generation

In [26]:
import openai
import os
import asyncio
import edge_tts
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# ─────────────────────────────────────────────────────
# Assumed defined upstream in your notebook: llm, db, client, gr, ChatPromptTemplate
# ─────────────────────────────────────────────────────

# ── Prompts ───────────────────────────────────────────────────────────────────

chatbot_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are a financial analyst at a stock firm. "
        "Answer questions related to the financial review politely for a prospective client.\n\n"
        "You can also generate images to visualize concepts if explicitly asked. "
        "Use the `generate_dalle_image` tool when the user explicitly asks to create or generate an image.\n\n"
        "Context:\n{context}"
    )),
    ("human", "Question: {question}"),
])

condense_question_prompt = ChatPromptTemplate.from_messages([
    ("system", "Given a chat history and a follow-up question, rephrase the follow-up question to be a standalone question."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
])

# ── Tool definitions ──────────────────────────────────────────────────────────

tools = [
    {
        "type": "function",
        "function": {
            "name": "generate_dalle_image",
            "description": (
                "Generate an image using gpt-image-1 based on a text prompt. "
                "Use this when the user explicitly asks to create or generate an image."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "visual_prompt": {
                        "type": "string",
                        "description": "A detailed text description to guide image generation.",
                    }
                },
                "required": ["visual_prompt"],
            },
        },
    },
]

# ── Core helpers ──────────────────────────────────────────────────────────────

def _build_langchain_history(history: list) -> list:
    """Convert Gradio dict-format history to LangChain message objects."""
    messages = []
    for turn in history:
        if isinstance(turn, dict):
            role = turn.get("role")
            content = turn.get("content", "")
            if role in ("user", "human") and content:
                messages.append(HumanMessage(content=content))
            elif role in ("assistant", "ai", "model") and content:
                messages.append(AIMessage(content=content))
    return messages


def generate_dalle_image(visual_prompt: str) -> str | None:
    """
    Generate an image via gpt-image-1.
    Returns a base64 data URI (or URL as fallback), or raises on failure.
    """
    print(f"[gpt-image-1] Generating image for: {visual_prompt}")
    try:
        response = client.images.generate(
            model="gpt-image-1",
            prompt=visual_prompt,
            size="1024x1024",
            n=1,
        )

        if not response or not response.data:
            raise ValueError("gpt-image-1 returned an empty response.")

        image_data = response.data[0]

        # gpt-image-1 returns base64, not a URL
        if hasattr(image_data, "b64_json") and image_data.b64_json:
            print("[gpt-image-1] Got base64 image data.")
            return f"data:image/png;base64,{image_data.b64_json}"

        # Fallback: try URL in case behaviour changes
        elif hasattr(image_data, "url") and image_data.url:
            print(f"[gpt-image-1] Got URL: {image_data.url}")
            return image_data.url

        else:
            raise ValueError("Response contained neither b64_json nor url.")

    except Exception as e:
        print(f"[gpt-image-1] Error: {e}")
        raise e


# ── RAG pipeline ──────────────────────────────────────────────────────────────

def ask_question(question: str, history: list | None = None) -> str:
    chat_history_for_llm = _build_langchain_history(history or [])

    standalone_question = question
    if chat_history_for_llm:
        try:
            condense_chain = condense_question_prompt | llm
            standalone_question = condense_chain.invoke({
                "question": question,
                "chat_history": chat_history_for_llm,
            }).content
        except Exception as e:
            print(f"[condense] Error: {e}")

    try:
        retrieved_docs = db.similarity_search(standalone_question, k=4)
    except NameError:
        retrieved_docs = vector_store.similarity_search(standalone_question, k=4)

    docs_content = "\n\n".join(
        doc.page_content for doc in retrieved_docs if hasattr(doc, "page_content")
    )

    messages = chatbot_prompt.format_messages(
        question=standalone_question,
        context=docs_content,
    )

    formatted_payload = []
    for msg in messages:
        role = "user" if msg.type == "human" else "system" if msg.type == "system" else "assistant"
        formatted_payload.append({"role": role, "content": msg.content})

    return llm.invoke(formatted_payload).content


# ── TTS / Transcription ───────────────────────────────────────────────────────

def transcribe_audio(audio_path: str) -> str:
    if not os.environ.get("OPENAI_API_KEY"):
        return "Error: OPENAI_API_KEY environment variable not set."
    with open(audio_path, "rb") as audio_file:
        transcript = client.audio.transcriptions.create(model="whisper-1", file=audio_file)
    return transcript.text


async def _tts(text: str) -> str:
    communicate = edge_tts.Communicate(text, "en-US-AndrewNeural")
    path = "response.mp3"
    await communicate.save(path)
    return path


# ── Text predict (with image generation) ─────────────────────────────────────

def predict_for_multimodal(message: str, history: list) -> tuple[list, str]:
    if history is None:
        history = []

    try:
        system_content = (
            "You are a financial analyst. "
            "If the user explicitly asks to generate or create an image, call the `generate_dalle_image` tool. "
            "Otherwise, answer their question directly."
        )
        langchain_msgs = [SystemMessage(content=system_content)]
        langchain_msgs.extend(_build_langchain_history(history))
        langchain_msgs.append(HumanMessage(content=message))

        llm_with_tools = llm.bind_tools(tools)
        ai_response = llm_with_tools.invoke(langchain_msgs)
        tool_calls = getattr(ai_response, "tool_calls", [])

        if tool_calls:
            tool_call = tool_calls[0]
            name = tool_call["name"] if isinstance(tool_call, dict) else tool_call.name
            args = tool_call["args"] if isinstance(tool_call, dict) else tool_call.args

            if name == "generate_dalle_image":
                visual_prompt = args.get("visual_prompt")
                if visual_prompt:
                    try:
                        url = generate_dalle_image(visual_prompt)
                        if url and url.startswith("data:image"):
                            # Render base64 image as raw HTML — Gradio won't render data URIs in markdown
                            reply = f'<img src="{url}" style="max-width:100%; border-radius:8px;"/>'
                        elif url:
                            reply = f"![Generated Image]({url})"
                        else:
                            reply = "Sorry, image generation failed. No image data was returned."
                    except Exception as dalle_error:
                        reply = f"Sorry, image generation failed: {dalle_error}"
                else:
                    reply = "I couldn't extract an image prompt from your request."
            else:
                reply = ask_question(message, history)
        else:
            reply = ai_response.content if ai_response.content else ask_question(message, history)

    except Exception as e:
        print(f"[predict_for_multimodal] Error: {e}")
        reply = ask_question(message, history)

    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": reply})
    return history, ""


# ── Voice handler ─────────────────────────────────────────────────────────────

def voice_chat_handler(audio_path: str, history: list) -> tuple[list, str | None, str]:
    if history is None:
        history = []

    if audio_path is None:
        return history, None, "No audio received. Please record your question."

    try:
        user_text = transcribe_audio(audio_path)
        if user_text.startswith("Error:"):
            history.append({"role": "assistant", "content": user_text})
            return history, None, user_text

        updated_history, _ = predict_for_multimodal(user_text, history)

        bot_reply = updated_history[-1]["content"] if updated_history else ""

        # Skip TTS for image replies (base64 HTML or markdown image)
        if bot_reply and not bot_reply.startswith("<img") and not bot_reply.startswith("!["):
            audio_output_path = asyncio.run(_tts(bot_reply))
            status = "Response generated successfully!"
        else:
            audio_output_path = None
            status = "Image generated successfully!"

        return updated_history, audio_output_path, status

    except Exception as e:
        error = f"Error in voice_chat_handler: {e}"
        print(error)
        history.append({"role": "user", "content": "[voice input]"})
        history.append({"role": "assistant", "content": error})
        return history, None, error


# ── Gradio UI ─────────────────────────────────────────────────────────────────

with gr.Blocks() as multimodal_app:
    gr.Markdown("## Multimodal RAG Assistant")

    # sanitize_html=False allows base64 <img> tags to render in the chatbot
    chatbot = gr.Chatbot(label="Chat History", sanitize_html=False)
    status_box = gr.Textbox(label="Voice Status", interactive=False, visible=True)
    audio_out = gr.Audio(label="🔊 Voice Response", autoplay=True)

    with gr.Row():
        msg = gr.Textbox(placeholder="Type a question or ask me to generate an image...", scale=4)
        audio_in = gr.Audio(sources="microphone", type="filepath", scale=1)

    msg.submit(
        predict_for_multimodal,
        inputs=[msg, chatbot],
        outputs=[chatbot, msg],
    )

    audio_in.stop_recording(
        voice_chat_handler,
        inputs=[audio_in, chatbot],
        outputs=[chatbot, audio_out, status_box],
    )

multimodal_app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c87f7488060535a598.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Can generate an image on the first prompt but when RAG is invoked to retrieve financial information from my dataset. Image generation stops

In [28]:
import openai
import os
import asyncio
import edge_tts
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# ─────────────────────────────────────────────────────
# Assumed defined upstream in your notebook: llm, db, client, gr, ChatPromptTemplate
# ─────────────────────────────────────────────────────

# ── Prompts ───────────────────────────────────────────────────────────────────

chatbot_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are a financial analyst at a stock firm. "
        "Answer questions related to the financial review politely for a prospective client.\n\n"
        "You can also generate images to visualize concepts if explicitly asked. "
        "Use the `generate_dalle_image` tool when the user explicitly asks to create or generate an image.\n\n"
        "Context:\n{context}"
    )),
    ("human", "Question: {question}"),
])

condense_question_prompt = ChatPromptTemplate.from_messages([
    ("system", "Given a chat history and a follow-up question, rephrase the follow-up question to be a standalone question."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
])

# ── Tool definitions ──────────────────────────────────────────────────────────

tools = [
    {
        "type": "function",
        "function": {
            "name": "generate_dalle_image",
            "description": (
                "Generate an image using gpt-image-1 based on a text prompt. "
                "Use this when the user explicitly asks to create or generate an image."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "visual_prompt": {
                        "type": "string",
                        "description": "A detailed text description to guide image generation.",
                    }
                },
                "required": ["visual_prompt"],
            },
        },
    },
]

# ── Core helpers ──────────────────────────────────────────────────────────────

def _build_langchain_history(history: list) -> list:
    """
    Convert Gradio history to LangChain message objects.
    Handles both dict-format {"role": ..., "content": ...}
    and tuple/list-format [user_msg, bot_msg] defensively.
    Base64 image strings are stripped to avoid token bloat.
    """
    messages = []
    for turn in history:
        if isinstance(turn, dict):
            role = turn.get("role")
            content = turn.get("content", "")
            # Skip image replies — base64 data URIs and markdown images
            if isinstance(content, str) and (content.startswith("<img") or content.startswith("![")):
                continue
            if role in ("user", "human") and content:
                messages.append(HumanMessage(content=content))
            elif role in ("assistant", "ai", "model") and content:
                messages.append(AIMessage(content=content))
        # Fallback: handle legacy tuple/list format Gradio may pass in some versions
        elif isinstance(turn, (list, tuple)) and len(turn) == 2:
            user_content = str(turn[0]) if turn[0] else ""
            bot_content  = str(turn[1]) if turn[1] else ""
            if user_content:
                messages.append(HumanMessage(content=user_content))
            # Strip image replies from tuple history too
            if bot_content and not bot_content.startswith("<img") and not bot_content.startswith("!["):
                messages.append(AIMessage(content=bot_content))
    return messages


def generate_dalle_image(visual_prompt: str) -> str | None:
    """
    Generate an image via gpt-image-1.
    Returns a base64 data URI (or URL as fallback), or raises on failure.
    """
    print(f"[gpt-image-1] Generating image for: {visual_prompt}")
    try:
        response = client.images.generate(
            model="gpt-image-1",
            prompt=visual_prompt,
            size="1024x1024",
            n=1,
        )

        if not response or not response.data:
            raise ValueError("gpt-image-1 returned an empty response.")

        image_data = response.data[0]

        # gpt-image-1 returns base64, not a URL
        if hasattr(image_data, "b64_json") and image_data.b64_json:
            print("[gpt-image-1] Got base64 image data.")
            return f"data:image/png;base64,{image_data.b64_json}"

        # Fallback: try URL in case behaviour changes
        elif hasattr(image_data, "url") and image_data.url:
            print(f"[gpt-image-1] Got URL: {image_data.url}")
            return image_data.url

        else:
            raise ValueError("Response contained neither b64_json nor url.")

    except Exception as e:
        print(f"[gpt-image-1] Error: {e}")
        raise e


# ── RAG pipeline ──────────────────────────────────────────────────────────────

def ask_question(question: str, history: list | None = None) -> str:
    chat_history_for_llm = _build_langchain_history(history or [])

    standalone_question = question
    if chat_history_for_llm:
        try:
            condense_chain = condense_question_prompt | llm
            standalone_question = condense_chain.invoke({
                "question": question,
                "chat_history": chat_history_for_llm,
            }).content
        except Exception as e:
            print(f"[condense] Error: {e}")

    try:
        retrieved_docs = db.similarity_search(standalone_question, k=4)
    except NameError:
        retrieved_docs = vector_store.similarity_search(standalone_question, k=4)

    docs_content = "\n\n".join(
        doc.page_content for doc in retrieved_docs if hasattr(doc, "page_content")
    )

    messages = chatbot_prompt.format_messages(
        question=standalone_question,
        context=docs_content,
    )

    formatted_payload = []
    for msg in messages:
        role = "user" if msg.type == "human" else "system" if msg.type == "system" else "assistant"
        formatted_payload.append({"role": role, "content": msg.content})

    return llm.invoke(formatted_payload).content


# ── TTS / Transcription ───────────────────────────────────────────────────────

def transcribe_audio(audio_path: str) -> str:
    if not os.environ.get("OPENAI_API_KEY"):
        return "Error: OPENAI_API_KEY environment variable not set."
    with open(audio_path, "rb") as audio_file:
        transcript = client.audio.transcriptions.create(model="whisper-1", file=audio_file)
    return transcript.text


async def _tts(text: str) -> str:
    communicate = edge_tts.Communicate(text, "en-US-AndrewNeural")
    path = "response.mp3"
    await communicate.save(path)
    return path


# ── Text predict (with image generation) ─────────────────────────────────────

def predict_for_multimodal(message: str, history: list) -> tuple[list, str]:
    if history is None:
        history = []

    try:
        system_content = (
            "You are a financial analyst. "
            "If the user explicitly asks to generate or create an image, call the `generate_dalle_image` tool. "
            "Otherwise, answer their question directly."
        )
        langchain_msgs = [SystemMessage(content=system_content)]
        langchain_msgs.extend(_build_langchain_history(history))
        langchain_msgs.append(HumanMessage(content=message))

        llm_with_tools = llm.bind_tools(tools)
        ai_response = llm_with_tools.invoke(langchain_msgs)
        tool_calls = getattr(ai_response, "tool_calls", [])

        if tool_calls:
            tool_call = tool_calls[0]
            name = tool_call["name"] if isinstance(tool_call, dict) else tool_call.name
            args = tool_call["args"] if isinstance(tool_call, dict) else tool_call.args

            if name == "generate_dalle_image":
                visual_prompt = args.get("visual_prompt")
                if visual_prompt:
                    try:
                        url = generate_dalle_image(visual_prompt)
                        if url and url.startswith("data:image"):
                            # Render base64 image as raw HTML — Gradio won't render data URIs in markdown
                            reply = f'<img src="{url}" style="max-width:100%; border-radius:8px;"/>'
                        elif url:
                            reply = f"![Generated Image]({url})"
                        else:
                            reply = "Sorry, image generation failed. No image data was returned."
                    except Exception as dalle_error:
                        reply = f"Sorry, image generation failed: {dalle_error}"
                else:
                    reply = "I couldn't extract an image prompt from your request."
            else:
                reply = ask_question(message, history)
        else:
            reply = ai_response.content if ai_response.content else ask_question(message, history)

    except Exception as e:
        print(f"[predict_for_multimodal] Error: {e}")
        reply = ask_question(message, history)

    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": reply})
    return history, ""


# ── Voice handler ─────────────────────────────────────────────────────────────

def voice_chat_handler(audio_path: str, history: list) -> tuple[list, str | None, str]:
    if history is None:
        history = []

    if audio_path is None:
        return history, None, "No audio received. Please record your question."

    try:
        user_text = transcribe_audio(audio_path)
        if user_text.startswith("Error:"):
            history.append({"role": "assistant", "content": user_text})
            return history, None, user_text

        updated_history, _ = predict_for_multimodal(user_text, history)

        bot_reply = updated_history[-1]["content"] if updated_history else ""

        # Skip TTS for image replies (base64 HTML or markdown image)
        if bot_reply and not bot_reply.startswith("<img") and not bot_reply.startswith("!["):
            audio_output_path = asyncio.run(_tts(bot_reply))
            status = "Response generated successfully!"
        else:
            audio_output_path = None
            status = "Image generated successfully!"

        return updated_history, audio_output_path, status

    except Exception as e:
        error = f"Error in voice_chat_handler: {e}"
        print(error)
        history.append({"role": "user", "content": "[voice input]"})
        history.append({"role": "assistant", "content": error})
        return history, None, error


# ── Gradio UI ─────────────────────────────────────────────────────────────────

with gr.Blocks() as multimodal_app:
    gr.Markdown("## Multimodal RAG Assistant")

    # type="messages" locks history to dict format; sanitize_html=False renders base64 <img> tags
    chatbot = gr.Chatbot(label="Chat History", sanitize_html=False)
    status_box = gr.Textbox(label="Voice Status", interactive=False, visible=True)
    audio_out = gr.Audio(label="🔊 Voice Response", autoplay=True)

    with gr.Row():
        msg = gr.Textbox(placeholder="Type a question or ask me to generate an image...", scale=4)
        audio_in = gr.Audio(sources="microphone", type="filepath", scale=1)

    msg.submit(
        predict_for_multimodal,
        inputs=[msg, chatbot],
        outputs=[chatbot, msg],
    )

    audio_in.stop_recording(
        voice_chat_handler,
        inputs=[audio_in, chatbot],
        outputs=[chatbot, audio_out, status_box],
    )

multimodal_app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2344fa57bcc9765e1d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
